# FacTora — Fake News Detection Model (RoBERTa) — V31
**Model**: `roberta-large`  
**Task**: Binary classification — Real (1) vs Fake (0)  
**Dataset**: 63,016 balanced articles  
**Target**: 85%+ accuracy  
**Changes from V30**: Swapped to RoBERTa-large to enable FP16 support and train in <4 hours.

In [ ]:
# Step 1: Environment Check
import torch
import transformers
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}       : {torch.cuda.get_device_name(i)} | BF16={torch.cuda.is_bf16_supported()}')
print(f'Transformers: {transformers.__version__}')

In [ ]:
# Step 2: Imports & Config
import os, gc, warnings, glob
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from torch import nn
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings('ignore')

# ── Hyperparameters ───────────────────────────────────────────────────────────
# V31 CONFIG:
#   1) Swap to roberta-large
#   2) BATCH_SIZE = 8, GRAD_ACCUM = 4, EPOCHS = 3
# ─────────────────────────────────────────────────────────────────────────────
MODEL_NAME   = 'roberta-large'
MAX_LEN      = 512
BATCH_SIZE   = 8
GRAD_ACCUM   = 4       # Effective batch = 32
EPOCHS       = 3
LR           = 1e-5    # Safe LR for RoBERTa large
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
SEED         = 42
OUTPUT_DIR   = '/kaggle/working/factora_roberta_model'

# DYNAMIC DATASET PATH FINDER
try:
    train_csv_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
    DATASET_PATH = os.path.dirname(train_csv_path)
    print(f'Found dataset at: {DATASET_PATH}')
except IndexError:
    print('ERROR: Could not find train.csv in /kaggle/input/')
    print('Contents of /kaggle/input:', os.listdir('/kaggle/input/') if os.path.exists('/kaggle/input/') else 'Directory not found')
    raise FileNotFoundError('Dataset missing. Please attach it.')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config: LR={LR}, BATCH={BATCH_SIZE}, GRAD_ACCUM={GRAD_ACCUM}, EPOCHS={EPOCHS}')

In [ ]:
# Step 3: Load Data
print('Loading data...')
train_df = pd.read_csv(os.path.join(DATASET_PATH, 'train.csv')).dropna(subset=['text','label'])
val_df   = pd.read_csv(os.path.join(DATASET_PATH, 'val.csv')).dropna(subset=['text','label'])
test_df  = pd.read_csv(os.path.join(DATASET_PATH, 'test.csv')).dropna(subset=['text','label'])

train_df['label'] = train_df['label'].astype(int)
val_df['label']   = val_df['label'].astype(int)
test_df['label']  = test_df['label'].astype(int)

print(f'Train : {len(train_df):,} | Real(1)={train_df["label"].sum():,} Fake(0)={(train_df["label"]==0).sum():,}')
print(f'Val   : {len(val_df):,}   | Real(1)={val_df["label"].sum():,} Fake(0)={(val_df["label"]==0).sum():,}')
print(f'Test  : {len(test_df):,}  | Real(1)={test_df["label"].sum():,} Fake(0)={(test_df["label"]==0).sum():,}')

In [ ]:
# Step 4: Tokenize
print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
        return_token_type_ids=False
    )

train_hf = HFDataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
val_hf   = HFDataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True))
test_hf  = HFDataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

train_hf = train_hf.map(tokenize, batched=True, batch_size=512, remove_columns=['text']).rename_column('label', 'labels')
val_hf   = val_hf.map(tokenize, batched=True, batch_size=512, remove_columns=['text']).rename_column('label', 'labels')
test_hf  = test_hf.map(tokenize, batched=True, batch_size=512, remove_columns=['text']).rename_column('label', 'labels')

train_hf.set_format('torch')
val_hf.set_format('torch')
test_hf.set_format('torch')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Tokenization complete.')

In [ ]:
# Step 5: Load model
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'FAKE', 1: 'REAL'},
    label2id={'FAKE': 0, 'REAL': 1},
    ignore_mismatched_sizes=True
)
# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model on device: {device}')

In [ ]:
# Step 6: Define metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average='weighted')
    unique, counts = np.unique(preds, return_counts=True)
    dist = dict(zip(unique.tolist(), counts.tolist()))
    print(f'  [Eval] pred_dist={dist} | acc={acc:.4f} | f1={f1:.4f}')
    return {'accuracy': acc, 'f1': f1}

In [ ]:
# Step 7: Configure Training
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    eval_strategy='epoch',
    save_strategy='epoch',
    greater_is_better=True,
    fp16=True,                       # ENABLE FP16 FOR SPEED
    bf16=False,                      
    logging_steps=50,
    dataloader_num_workers=2,
    save_total_limit=2,
    seed=SEED,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print('Trainer configured. Starting training...')

In [ ]:
# Step 8: TRAIN
train_result = trainer.train()
print('\n=== Training Complete ===')
print(f'Total steps  : {train_result.global_step}')
print(f'Train loss   : {train_result.training_loss:.4f}')

# Save the best model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Best model saved to {OUTPUT_DIR}')

In [ ]:
# Step 9: Final evaluation on test set
print('\n=== Final Test Set Evaluation ===')
test_results = trainer.predict(test_hf)
test_preds   = np.argmax(test_results.predictions, axis=-1)
test_labels  = test_results.label_ids

acc = accuracy_score(test_labels, test_preds)
f1  = f1_score(test_labels, test_preds, average='weighted')

print(f'\nTest Accuracy : {acc:.4f} ({acc*100:.2f}%)')
print(f'Test F1 Score : {f1:.4f}')
print(f'\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=['FAKE (0)', 'REAL (1)']))
print(f'\nConfusion Matrix:')
print(confusion_matrix(test_labels, test_preds))

In [ ]:
# Step 10: Save training history as CSV for review
history_df = pd.DataFrame(trainer.state.log_history)
history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)
print('Training history saved.')
print(history_df.tail(10).to_string())